<a href="https://colab.research.google.com/github/vaishnavi2810-code/AI-For-Beginners/blob/main/shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interpreting Classifier Weights

In this experiment, you will train models to distringuish examples of two different genres of Shakespeare's plays: comedies and tragedies. (We'll ignore the histories, sonnets, etc.) Since he died four hundred years ago, Shakespeare has not written any more plays—although scraps of various other works have come to light. We are not, therefore, interested in building models simply to help categorize an unbounded stream of future documents, as we might be in other applications of text classification; rather, we are interested in what a classifier might have to tell us about what we mean by the terms “comedy” and “tragedy”.

You will start by copying and running your `createBasicFeatures` function from the experiment with movie reviews. Do the features the classifier focuses on tell you much about comedy and tragedy in general?

You will then implement another featurization function `createInterestingFeatures`, which will focus on only those features you think are informative for distinguishing between comedy and tragedy. Accuracy on leave-one-out cross-validation may go up, but it more important to look at the features given the highest weight by the classifier. Interpretability in machine learning, of course, may be harder to define than accuracy—although accuracy at some tasks is hard enoough.

In [10]:
import json
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate,LeaveOneOut
import numpy as np

In [11]:
#read in the shakespeare corpus
def readShakespeare():
  raw = requests.get("https://raw.githubusercontent.com/dasmiq/cs6120-assignment2/refs/heads/main/shakespeare_plays.json").text.strip()
  corpus = [json.loads(line) for line in raw.split("\n")]

  #remove histories from the data, as we're only working with tragedies and comedies
  corpus = [entry for entry in corpus if entry["genre"] != "history"]
  return corpus

This is where you will implement two functions to featurize the data:

In [12]:
# TODO: Implement createBasicFeatures
# NB: The current contents are for testing only
# This function should return:
#  -a sparse numpy matrix of document features
#  -a list of the correct genre for each document
#  -a list of the vocabulary used by the features, such that the ith term of the
#    list is the word whose counts appear in the ith column of the matrix.

# This function should create a feature representation using all tokens that
# contain an alphabetic character.
from sklearn.feature_extraction.text import CountVectorizer
import re
def createBasicFeatures(corpus):
  #Your code here
  texts = [doc["text"] for doc in corpus]
  classes = [doc["genre"] for doc in corpus]

  # Define token pattern: keep tokens with at least one alphabetic character
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"

  # Use CountVectorizer to build the matrix
  vectorizer = CountVectorizer(token_pattern=token_pattern, lowercase=True)
  X = vectorizer.fit_transform(texts)

  # Get vocab list in the correct column order. It tells us which word is there corresponding to each column.
  vocab = vectorizer.get_feature_names_out().tolist()
  return X,classes,vocab

In [13]:
# TODO: Implement createInterestingFeatures. Describe your features and what
# they might tell you about the difference between comedy and tragedy.
# This function can add other features you want that help classification
# accuracy, such as bigrams, word prefixes and suffixes, etc.
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
def createInterestingFeatures(corpus):
  #Your code here
  texts = [doc["text"] for doc in corpus]
  genres = [doc["genre"] for doc in corpus]

  # --- TF-IDF unigrams + bigrams ---
  token_pattern = r"(?u)\b\w*[A-Za-z]\w*\b"
  vectorizer = TfidfVectorizer(token_pattern=token_pattern,lowercase=True, ngram_range=(1,3),min_df=2)
  X_tfidf = vectorizer.fit_transform(texts)

  vocab = vectorizer.get_feature_names_out().tolist()

  # --- 2. Document length ---
  doc_lengths = np.array([len(t.split()) for t in texts]).reshape(-1, 1)
  vocab += ["doc_length"]

  # --- 3. Archaic word count ---
  archaic = {"thou","thee","thy","thine","ye","hast","hath","dost"}
  archaic_count = np.array([sum(1 for w in t.lower().split() if w in archaic) for t in texts]).reshape(-1,1)
  vocab += ["archaic_count"]

  # --- 4. Tragedy-related word count ---
  tragedy_words = {"death","die","dies","died","blood","murder","kill","grave","woe","ghost"}
  tragedy_count = np.array([sum(1 for w in re.findall(r"(?u)\b\w+[A-Za-z]\w*\b", t.lower()) if w in tragedy_words) for t in texts]).reshape(-1,1)
  vocab += ["tragedy_word_count"]

  # --- 5. Comedy-related word count ---
  comedy_words = {"love","marry","fool","merry","happy","joy","wedding"}
  comedy_count = np.array([sum(1 for w in re.findall(r"(?u)\b\w+[A-Za-z]\w*\b", t.lower()) if w in comedy_words) for t in texts]).reshape(-1,1)
  vocab += ["comedy_word_count"]

  # --- 6. Uppercase title word count ---
  title_words = {"KING","QUEEN","LORD","PRINCE","DUKE","HAMLET","MACBETH"}
  def uppercase_token_count(s):
    tokens = re.findall(r"\b[A-Z]{2,}\b", s)
    return sum(1 for tok in tokens if tok in title_words)
  title_count = np.array([uppercase_token_count(t) for t in texts]).reshape(-1,1)
  vocab += ["title_word_count"]

  # --- Combine all features ---
  X = hstack([X_tfidf, doc_lengths, archaic_count, tragedy_count, comedy_count, title_count])

  return X, genres, vocab

In [14]:
#given a numpy matrix representation of the features for the training set, the
# vector of true classes for each example, and the vocabulary as described
# above, this computes the accuracy of the model using leave one out cross
# validation and reports the most indicative features for each class
def evaluateModel(X,y,vocab,penalty="l1"):
  #create and fit the model
  model = LogisticRegression(penalty=penalty,solver="liblinear")
  results = cross_validate(model,X,y,cv=LeaveOneOut())

  #determine the average accuracy
  scores = results["test_score"]
  avg_score = sum(scores)/len(scores)

  #determine the most informative features
  # this requires us to fit the model to everything, because we need a
  # single model to draw coefficients from, rather than 26
  model.fit(X,y)
  neg_class_prob_sorted = model.coef_[0, :].argsort()
  pos_class_prob_sorted = (-model.coef_[0, :]).argsort()

  termsToTake = 20
  pos_indicators = [vocab[i] for i in neg_class_prob_sorted[:termsToTake]]
  neg_indicators = [vocab[i] for i in pos_class_prob_sorted[:termsToTake]]

  return avg_score,pos_indicators,neg_indicators

def runEvaluation(X,y,vocab):
  print("----------L1 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l1")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)
  #this call will fit a model with L2 normalization
  print("----------L2 Norm-----------")
  avg_score,pos_indicators,neg_indicators = evaluateModel(X,y,vocab,"l2")
  print("The model's average accuracy is %f"%avg_score)
  print("The most informative terms for pos are: %s"%pos_indicators)
  print("The most informative terms for neg are: %s"%neg_indicators)


In [15]:
corpus = readShakespeare()

Run the following to train and evaluate two models with basic features:

In [16]:
X,y,vocab = createBasicFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.615385
The most informative terms for pos are: ['you', 'helena', 'prospero', 'duke', 'sir', 'i', 'leontes', 'a', 'of', 'presenting', 'preserver', 'preserved', 'pretty', 'prettiness', 'prettily', 'prettiest', 'prettier', 'presentment', 'presently', 'preservers']
The most informative terms for neg are: ['him', 's', 'iago', 'imogen', 'o', 'brutus', 'lear', 'ham', 'and', 'what', 'rom', 'the', 'presence', 'prettier', 'pretia', 'pretext', 'pretense', 'pretending', 'presenters', 'presented']
----------L2 Norm-----------
The model's average accuracy is 0.769231
The most informative terms for pos are: ['i', 'you', 'duke', 'prospero', 'a', 'helena', 'your', 'antonio', 'sir', 'leontes', 'hermia', 'for', 'lysander', 'ariel', 'sebastian', 'demetrius', 'camillo', 'stephano', 'me', 'parolles']
The most informative terms for neg are: ['iago', 'othello', 's', 'him', 'imogen', 'what', 'lear', 'brutus', 'his', 'cassio', 'o', 'ham', 'our', 'de

Run the following to train and evaluate two models with features that are interesting for distinguishing comedy and tragedy:

In [17]:
X,y,vocab = createInterestingFeatures(corpus)
runEvaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.692308
The most informative terms for pos are: ['comedy_word_count', 'doc_length', 's spirit', 's such', 's strength', 's strange', 's storm', 's steward', 's statue which', 's statue', 's state', 's staff', 's spring', 's spoken', 's such a', 's spent', 's speed', 's sound', 's soul', 's sorrow']
The most informative terms for neg are: ['tragedy_word_count', 'archaic_count', 's speed', 's strange', 's storm', 's steward', 's statue which', 's statue', 's state', 's staff', 's spring', 's spoken', 's spirit', 's spent', 's strength', 's sound', 's soul', 's sorrow', 's sons', 's song come']
----------L2 Norm-----------
The model's average accuracy is 0.730769
The most informative terms for pos are: ['i', 'of', 'a', 'angelo', 'and', 'you', 'to', 'the', 'duke', 'lucio', 'my', 'me', 'for', 'it', 'your', 'stephano', 'sir', 'that', 'sebastian', 'd']
The most informative terms for neg are: ['troilus', 'cressida', 'pandarus', 'ach

**TODO**: Based on the most informative features in the output of the classifier evaluation, what do these classifiers tell you about the differences between comedy and tragedy?

1. Basic Features Model (Unigrams):Accuracy: L1 ≈ 61.5%, L2 ≈ 76.9%
Character names like "duke," "prospero," "helena," and "leontes," as
*   Character names like "duke," "prospero," "helena," and "leontes," as well as pronouns like "i" and "you," are positive (comedy) markers. Characters like "Iago," "Ham," "Lear," "Brutus," and "Othello" are examples of negative (tragic) signs.
* Interpretation: To differentiate between genres, the model mostly uses pronouns and character names.
Tragedies have tragic heroes or villains, whereas comedies frequently feature characters connected to romantic or comic themes.
L2 distributes weight across more characters, while L1 chooses the most unique character names.


2. Handcrafted + N-grams Interesting Features Model: Accuracy: L1 ≈ 69.2%, L2 ≈ 73.1%
* 'comedy_count', 'doc_length', and terms like 'rome renowned' and 'rome but' are examples of positive (comedy) indications.
'death_count', 'archaic_count', and repeated bigrams like 'rome or' and 'rome before' are examples of negative (traumatic) signs.
*   Interpretation: Beyond only a few words, the engineered features offer insight:
* 'comedy_count' → counts phrases that are specific to comedy directly.
* "death_count" is frequently used in disasters.
* "archaic_count" refers to more formal or older terminology, frequently found in tragedies.
* When comparing comedies and tragedies, Bigrams captures similar phrases or patterns (e.g., plot-related language).
The length of the document also matters; dramas typically have longer, narrative-heavy sections, while comedies may have shorter, more dialogue-heavy language.


3. Three Crucial Takeaways:
* Shorter text passages, specific character names, and lighter, more fun or amorous language are characteristics of comedies.
* Death-related terminology, formal or archaic language, and the existence of tragic heroes or villains are characteristics of tragedies.
* Even if handcrafted features do not significantly outperform unigrams, they do improve interpretability and somewhat increase accuracy by capturing structural and thematic patterns.
